# Fine-tune the city-directory extractor — free Colab path

**No paid HF plan needed.** This trains `train/sft_qwen.py` on a free **T4** and pushes the
result to your (free) Hugging Face namespace.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
2. Replace `<you>` (your GitHub repo) and `YOUR_HF_USERNAME` below.
3. 0.8B / 2B fit the T4 directly; for **4B add `--qlora`** (4-bit). Precision auto-selects fp16 on T4.

In [ ]:
!pip install -q "trl>=0.12" "transformers>=4.46" "datasets>=3.0" "accelerate>=1.0" "peft>=0.13" "bitsandbytes>=0.43"

In [ ]:
# free token (write scope) from https://huggingface.co/settings/tokens — only needed for --push-to-hub
from huggingface_hub import login
login()

In [ ]:
!git clone https://github.com/<you>/city-directory-extraction.git
%cd city-directory-extraction

In [ ]:
# synthetic training data (license-clean, ours) — ~100k line->record pairs
!python data_prep/synth_persons.py --n 100000 --out data/synth_train.jsonl --seed 13

## Train + push
2B LoRA is the recommended first run (~20–40 min on a T4). Swap `--model` for 0.8B; for **4B add `--qlora`**.
`--target` (pipe/yaml) here MUST match what you later pass to `eval/qwen_predict.py`.

In [ ]:
!python train/sft_qwen.py --train-file data/synth_train.jsonl --model Qwen/Qwen3.5-2B --hub-model-id YOUR_HF_USERNAME/city-directory-extractor-2b --target pipe --push-to-hub
# 4B on the free T4: append  --qlora

## Smoke-test the loop (self-contained, no downloads)
Tiny synthetic dev set → predict with the model you just pushed → score. Proves train→predict→score for $0.
Default training is **LoRA**, so `qwen_predict.py` needs `--base-model` (the adapter's base).
For the real NYU/FTD/Tulsa eval sets, run `eval/qwen_predict.py` + `eval/evaluate.py` locally per the README.

In [ ]:
!python data_prep/synth_persons.py --n 200 --out data/synth_dev.jsonl --seed 99
!python eval/qwen_predict.py --base-model Qwen/Qwen3.5-2B --model YOUR_HF_USERNAME/city-directory-extractor-2b --gold data/synth_dev.jsonl --target pipe
!python eval/evaluate.py --gold data/synth_dev.jsonl --pred data/preds_qwen.txt --target pipe